# Guest Topics — Last 3 Months
**Input:** `2026_04_17_messages.xls` (all 3 sheets)
**Output:**
- `guest_topics_3m.parquet` — top-10 topics per hotel (for bar chart)
- `thread_topics_3m.parquet` — per-thread topic labels (for drill-down table)

---
### Метод
1. Загружаем все 3 листа XLS, фильтруем последние 3 месяца
2. Разбиваем на треды по паузе > 4ч (только гостевые сообщения)
3. GPT-4o-mini классифицирует каждый тред → одна тема из фиксированного списка (батчами по 15)
4. Агрегируем топ-10 тем по каждому отелю

In [1]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import json, re
from pathlib import Path
from datetime import timedelta
from tqdm.auto import tqdm
from openai import OpenAI

DATA_DIR    = Path('/home/nikita/code/PlatoIsDead/notebook_prototype/sentiment_analysis/data')
MSG_PATH    = DATA_DIR / '2026_04_17_messages.xls'
OUT_TOPICS  = DATA_DIR / 'guest_topics_3m.parquet'
OUT_THREADS = DATA_DIR / 'thread_topics_3m.parquet'

HOTEL_NAMES   = {1: 'БС74', 2: 'МК16', 3: 'Дубай', 4: 'М73', 5: 'О-44'}
GAP_HOURS     = 4
LOOKBACK_DAYS = 90
BATCH_SIZE    = 15
MODEL         = 'gpt-4o-mini'

client = OpenAI()  # uses OPENAI_API_KEY env var
print('Ready.')

Ready.


In [2]:
# ── Cell 1: Load all 3 sheets ─────────────────────────────────────────────────
COLS = ['booking_id', 'hotel_id', 'message', 'from_admin',
        'date_add', 'is_admin_read', 'is_user_read']

xl = pd.ExcelFile(MSG_PATH, engine='xlrd')
print(f'Sheets: {xl.sheet_names}')

parts = []
for i, sheet in enumerate(xl.sheet_names):
    if i == 0:
        df = pd.read_excel(MSG_PATH, sheet_name=sheet, engine='xlrd')
    else:
        df = pd.read_excel(MSG_PATH, sheet_name=sheet, engine='xlrd',
                           header=None, names=COLS)
    print(f"  '{sheet}': {len(df):,} rows")
    parts.append(df)

msg = pd.concat(parts, ignore_index=True)
msg = msg.drop_duplicates(subset=['booking_id', 'date_add', 'message', 'from_admin'])

msg['date_add']   = pd.to_datetime(msg['date_add'])
msg['is_admin']   = msg['from_admin'].fillna(0).astype(int)
msg['message']    = msg['message'].astype(str).str.strip()
msg = msg.rename(columns={'booking_id': 'ID_booking'})
msg['hotel_name'] = msg['hotel_id'].map(HOTEL_NAMES)

SKIP = r'^(nan|тест|test|бегу|file|\s*)$|^<div'
msg = msg[~msg['message'].str.lower().str.match(SKIP)].copy()
msg = msg[msg['message'].str.len() >= 3].copy()
msg = msg.sort_values(['ID_booking', 'date_add']).reset_index(drop=True)

print(f'\nВсего строк:  {len(msg):,}')
print(f'Период:       {msg["date_add"].min().date()} → {msg["date_add"].max().date()}')
print(f'Отели:        {[HOTEL_NAMES[h] for h in sorted(msg["hotel_id"].unique())]}')

Sheets: ['messages', 'messages(2)', 'messages(3)']
  'messages': 65,535 rows
  'messages(2)': 65,536 rows
  'messages(3)': 57,205 rows

Всего строк:  143,085
Период:       2022-11-01 → 2026-04-17
Отели:        ['БС74', 'МК16', 'Дубай', 'М73', 'О-44']


In [3]:
# ── Cell 2: Filter to last 3 months, guest messages only ─────────────────────
cutoff = msg['date_add'].max() - timedelta(days=LOOKBACK_DAYS)
recent = msg[msg['date_add'] >= cutoff].copy()
guest  = recent[recent['is_admin'] == 0].copy()

print(f'Cutoff: {cutoff.date()}')
print(f'Период фильтра: {recent["date_add"].min().date()} → {recent["date_add"].max().date()}')
print()
print(recent.groupby('hotel_name').agg(
    all_msgs  = ('message', 'size'),
    bookings  = ('ID_booking', 'nunique'),
    guest_msg = ('is_admin', lambda x: (x == 0).sum()),
    admin_msg = ('is_admin', lambda x: (x == 1).sum()),
))

Cutoff: 2026-01-17
Период фильтра: 2026-01-17 → 2026-04-17

            all_msgs  bookings  guest_msg  admin_msg
hotel_name                                          
БС74             779       123        509        270
Дубай            734        91        397        337
М73               91        17         59         32
МК16             263       129         96        167
О-44           16876      1415       9852       7024


In [4]:
# ── Cell 3: Thread detection ──────────────────────────────────────────────────
recent_s = recent.sort_values(['ID_booking', 'date_add']).reset_index(drop=True)
recent_s['prev_time']  = recent_s.groupby('ID_booking')['date_add'].shift(1)
recent_s['gap_hrs']    = (recent_s['date_add'] - recent_s['prev_time']).dt.total_seconds() / 3600
recent_s['new_thread'] = recent_s['prev_time'].isna() | (recent_s['gap_hrs'] > GAP_HOURS)
recent_s['thread_id']  = recent_s.groupby('ID_booking')['new_thread'].cumsum()

guest_s = recent_s[recent_s['is_admin'] == 0].copy()

threads = (
    guest_s
    .groupby(['hotel_id', 'hotel_name', 'ID_booking', 'thread_id'])
    .agg(
        text         = ('message', lambda msgs: ' | '.join(msgs)),
        n_guest_msgs = ('message', 'count'),
        thread_start = ('date_add', 'min'),
    )
    .reset_index()
)

print(f'Тредов для классификации: {len(threads):,}')
print(threads.groupby('hotel_name')['thread_id'].count().rename('threads'))

Тредов для классификации: 3,962
hotel_name
БС74      245
Дубай     166
М73        27
МК16       61
О-44     3463
Name: threads, dtype: int64


In [5]:
# ── Cell 4: GPT Topic Classification ─────────────────────────────────────────
TOPICS = [
    'Wi-Fi / интернет',
    'Уборка / клининг',
    'Кондиционер / температура',
    'Заселение / выезд',
    'Ремонт / поломка',
    'Оплата / депозит',
    'Шум / соседи',
    'Парковка',
    'Прачечная',
    'Код / пароль / ключ',
    'Вопрос об объекте',
    'Переезд / смена номера',
    'Техника и приборы',
    'Питание',
    'Другое',
]

TOPIC_LIST = '\n'.join(f'- {t}' for t in TOPICS)

SYSTEM_PROMPT = f"""Ты — аналитик гостиничных обращений. Для каждого треда гостевых сообщений выбери ОДНУ тему из списка ниже.

Темы:
{TOPIC_LIST}

Верни ТОЛЬКО JSON-массив (без markdown, без пояснений):
[{{"idx": <int>, "topic": "<точно как в списке выше>", "confidence": <0.0-1.0>}}]""".strip()


def label_batch(batch_df):
    items = []
    for i, row in enumerate(batch_df.itertuples()):
        items.append(f'[{i}] {row.text[:300]}')
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': '\n\n'.join(items)},
        ],
    )
    raw = resp.choices[0].message.content.strip()
    raw = re.sub(r'^```[a-z]*\n?|\n?```$', '', raw).strip()
    return json.loads(raw)


n           = len(threads)
topics      = ['Другое'] * n
confidences = [0.0] * n

for start in tqdm(range(0, n, BATCH_SIZE)):
    batch = threads.iloc[start: start + BATCH_SIZE]
    try:
        results = label_batch(batch)
        for r in results:
            global_idx = start + int(r['idx'])
            if 0 <= global_idx < n:
                topics[global_idx]      = r.get('topic', 'Другое')
                confidences[global_idx] = float(r.get('confidence', 0.8))
    except Exception as e:
        print(f'  ⚠️  Batch {start}–{start + BATCH_SIZE} failed: {e}')

threads['topic']      = topics
threads['confidence'] = confidences

print(threads['topic'].value_counts())

  0%|          | 0/265 [00:00<?, ?it/s]

topic
Уборка / клининг             846
Прачечная                    521
Другое                       424
Ремонт / поломка             384
Wi-Fi / интернет             341
Оплата / депозит             310
Техника и приборы            211
Код / пароль / ключ          206
Заселение / выезд            193
Вопрос об объекте            176
Шум / соседи                 141
Кондиционер / температура     72
Питание                       61
Переезд / смена номера        47
Парковка                      28
Безопасность                   1
Name: count, dtype: int64


In [6]:
# ── Cell 5: Aggregate top-10 topics per hotel ─────────────────────────────────
hotel_totals = threads.groupby('hotel_name').size().rename('hotel_total')

topic_counts = (
    threads
    .groupby(['hotel_name', 'topic'])
    .agg(
        n_threads = ('thread_id', 'count'),
        example   = ('text', lambda x: x.iloc[0][:150]),
    )
    .reset_index()
    .join(hotel_totals, on='hotel_name')
)
topic_counts['pct_of_hotel'] = (
    topic_counts['n_threads'] / topic_counts['hotel_total'] * 100
).round(1)

top10 = (
    topic_counts
    .sort_values(['hotel_name', 'n_threads'], ascending=[True, False])
    .groupby('hotel_name', group_keys=False)
    .head(10)
    .drop(columns='hotel_total')
    .reset_index(drop=True)
)

print(top10.to_string())

   hotel_name                      topic  n_threads                                                                                                                                                 example  pct_of_hotel
0        БС74           Уборка / клининг         66  где парковка? | Подскажите контакты | А контакты отеля? | У вас есть генеральная уборка? | Есть ли генеральная уборка? | У меня трио номер, какая цена          26.9
1        БС74           Wi-Fi / интернет         31                                               Добрый вечер. Дайте, пожалуйста, 3 ваучера на вайфай, не проходит регистрация по номеру | Спасибо большое          12.7
2        БС74                     Другое         27                                                                                                                                                 Bonsoir          11.0
3        БС74           Ремонт / поломка         22                                                                             

In [7]:
# ── Cell 6: Save parquets ─────────────────────────────────────────────────────
top10.to_parquet(OUT_TOPICS, index=False)

threads[['hotel_id', 'hotel_name', 'ID_booking', 'thread_id',
         'topic', 'text', 'thread_start', 'n_guest_msgs', 'confidence']].to_parquet(
    OUT_THREADS, index=False)

print(f'✓  {OUT_TOPICS.name:<30} {len(top10):>5} rows  {OUT_TOPICS.stat().st_size/1024:.0f} KB')
print(f'✓  {OUT_THREADS.name:<30} {len(threads):>5} rows  {OUT_THREADS.stat().st_size/1024:.0f} KB')

✓  guest_topics_3m.parquet           50 rows  8 KB
✓  thread_topics_3m.parquet        3962 rows  543 KB
